# COCO bicycle eligibility count
Count unique source images from the mounted COCO 2017 train and validation data. Exclude all 100 prior development sources and every image with any bicycle crowd annotation, even when it also has ordinary bicycle annotations. A negative has no bicycle annotation. Inventory image filenames against the mounted split directories. Reading only filenames and annotations is sufficient for this count; no JPEGs are decoded or exported. No experimental splits are selected.

In [ ]:
import gc
import json
import os
from collections import Counter
from pathlib import Path
from time import perf_counter

start = perf_counter()
development = set([5001,7795,8211,8899,22623,30504,36844,40083,46872,65350,70774,82821,87038,89697,97988,101762,102411,109055,117645,123213,129113,130386,132375,135670,142585,144333,150417,152771,153527,157756,162858,164115,169996,184324,190753,193926,195918,196843,210273,219271,223738,226417,228436,244411,245173,252701,254814,263463,274687,277005,281929,289415,301061,306437,309391,319607,350003,370208,378139,384527,389804,390555,394328,394559,403565,411754,414340,414510,416991,421060,425227,428111,449996,451043,458223,460229,460682,468245,468501,470173,471893,476704,479912,487583,490936,491213,494913,502347,507797,512836,514376,534664,537672,545594,546823,550691,554328,565624,573094,575357])
assert len(development) == 100
mount = Path("/kaggle/input")
candidates = [mount / "coco-2017-dataset/coco2017", mount / "datasets/awsaf49/coco-2017-dataset/coco2017"]
root = next((p for p in candidates if (p / "annotations/instances_val2017.json").is_file()), None)
if root is None:
    found = next(mount.rglob("instances_val2017.json"), None)
    root = found.parent.parent if found else None

import requests
import shutil
import zipfile
import fsspec
from tempfile import TemporaryDirectory
temporary = TemporaryDirectory()
annotation_dir = root / "annotations" if root else Path(temporary.name)
base_url = "https://s3.amazonaws.com/images.cocodataset.org"
if root is None:
    archive = annotation_dir / "annotations.zip"
    with requests.get(base_url + "/annotations/annotations_trainval2017.zip", stream=True, timeout=120) as response:
        response.raise_for_status()
        with archive.open("wb") as f:
            for chunk in response.iter_content(1024*1024):
                f.write(chunk)
    with zipfile.ZipFile(archive) as z:
        for split in ("train2017", "val2017"):
            filename = f"instances_{split}.json"
            with z.open(f"annotations/{filename}") as src, (annotation_dir / filename).open("wb") as dst:
                shutil.copyfileobj(src, dst)

def available_filenames(split):
    if root is not None and (root / split).is_dir():
        with os.scandir(root / split) as entries:
            return {entry.name for entry in entries}, "Kaggle mounted image directory"
    # zipfile seeks to the central directory; only HTTP range blocks are read.
    with fsspec.open(base_url + f"/zips/{split}.zip", "rb", block_size=1024*1024) as online:
        with zipfile.ZipFile(online) as z:
            return {Path(name).name for name in z.namelist() if name.endswith(".jpg")}, "official COCO ZIP index"


def inventory(split):
    with (annotation_dir / f"instances_{split}.json").open() as f:
        data = json.load(f)
    category, = [c["id"] for c in data["categories"] if c["name"] == "bicycle"]
    images = {im["id"]: im["file_name"] for im in data["images"]}
    assert len(images) == len(data["images"])
    bicycle_records = [a for a in data["annotations"] if a["category_id"] == category]
    del data
    gc.collect()
    crowd_sources = {a["image_id"] for a in bicycle_records if a["iscrowd"]}
    annotated = {a["image_id"] for a in bicycle_records}
    regular = Counter(a["image_id"] for a in bicycle_records if not a["iscrowd"])
    available, verification = available_filenames(split)
    print(split, verification, flush=True)
    listed = set(images)
    assert annotated <= listed
    missing = {i for i, name in images.items() if name not in available}
    dev = listed & development
    eligible = listed - dev - crowd_sources - missing
    positives = eligible & annotated
    negatives = eligible - annotated
    assert positives == eligible.intersection(regular)
    assert len(positives) + len(negatives) + len(dev | crowd_sources | missing) == len(listed)
    return listed, {
        "sources_listed":len(listed), "unavailable_filenames":len(missing),
        "bicycle_crowd_sources":len(crowd_sources), "development_sources":len(dev),
        "unique_excluded":len(dev | crowd_sources | missing),
        "eligible_positive":len(positives), "eligible_negative":len(negatives),
        "eligible_positive_bicycle_instances":sum(regular[i] for i in positives),
        "eligible_total":len(eligible)}

seen = set()
report = {"annotation_source": str(root) if root else base_url, "splits": {}}
for split in ("train2017", "val2017"):
    source_ids, counts = inventory(split)
    assert seen.isdisjoint(source_ids)
    seen.update(source_ids)
    report["splits"][split] = counts
    print(split, counts, flush=True)
assert development <= seen
report["totals"] = dict(sum((Counter(row) for row in report["splits"].values()), Counter()))
pos = report["totals"]["eligible_positive"]
neg = report["totals"]["eligible_negative"]
report["maximum_balanced_sources"] = 2 * min(pos, neg)
report["shortfall_for_3500_positives"] = max(0, 3500-pos)
report["runtime_seconds"] = perf_counter() - start
Path("/kaggle/working/eligibility_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))